# RoSE on ChartQAPro — Full Pipeline
**Runtime:** GPU (T4 free tier)  
**Time:** ~3 hours for all 1,948 questions  
**Do first:** Runtime → Change runtime type → T4 GPU

This notebook auto-saves to Google Drive every 50 questions.  
If Colab disconnects, just re-run from Cell 1 — it resumes automatically.

## Cell 1 — Mount Google Drive (run this FIRST every session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# All work lives here — survives Colab disconnects
WORK_DIR = '/content/drive/MyDrive/RoSE_ChartQAPro'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

## Cell 2 — Install packages (run once, skip on resume)

In [ ]:
# Check if already installed
import importlib
needs_install = not importlib.util.find_spec('transformers')

if needs_install:
    print('Installing packages...')
    !pip install -q transformers==4.47.0 \
        accelerate bitsandbytes \
        sentence-transformers \
        qwen-vl-utils \
        Pillow tqdm
    print('✓ Done')
else:
    print('✓ Packages already installed')

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 3 — Clone repos (run once)

In [ ]:
from pathlib import Path

# Clone ChartQAPro dataset
if not Path('ChartQAPro').exists():
    print('Cloning ChartQAPro dataset...')
    !git clone https://github.com/vis-nlp/ChartQAPro.git
    print('✓ ChartQAPro cloned')
else:
    print('✓ ChartQAPro already present')

# Clone your RoSE repo (for reference / logging)
if not Path('RoSE').exists():
    print('Cloning your RoSE repo...')
    !git clone https://github.com/ktahsinr/RoSE.git
    print('✓ RoSE repo cloned')
else:
    print('✓ RoSE repo already present')

# Create results dir
Path('results').mkdir(exist_ok=True)

# Quick check on dataset
import json
with open('ChartQAPro/data/test.json') as f:
    data = json.load(f)
print(f'\nDataset loaded: {len(data)} questions')

# Show question type distribution
from collections import Counter
types = Counter(d.get('question_type','unknown') for d in data)
print('\nQuestion types:')
for qt, count in types.most_common():
    print(f'  {qt:<20} {count}')

## Cell 4 — Save the RoSE-CQA implementation script

In [ ]:
# Write the full implementation to disk
# (Copy your rose_chartqapro.py content here, or upload it from Drive)

# Option A: Upload from your computer
# from google.colab import files
# files.upload()   # upload rose_chartqapro.py

# Option B: It's already in your Drive (if you saved it there)
import shutil
src = Path('/content/drive/MyDrive/rose_chartqapro.py')
if src.exists():
    shutil.copy(src, 'rose_chartqapro.py')
    print('✓ Copied rose_chartqapro.py from Drive')
else:
    print('⚠ Upload rose_chartqapro.py to your Drive root first.')
    print('  Then re-run this cell.')

## Cell 5 — Load model (run once per session)

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'Qwen/Qwen2.5-VL-3B-Instruct'

print('Loading Qwen2.5-VL-3B (4-bit quantized)...')
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()
print('✓ VLM loaded')

print('Loading embedding model...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print('✓ Embedding model loaded')

# Memory check
allocated = torch.cuda.memory_allocated() / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\nGPU memory used: {allocated:.1f} / {total:.1f} GB')

## Cell 6 — Run RoSE on all 1,948 questions

In [ ]:
# Import and run
# The script auto-resumes if checkpoint.json exists

import importlib, rose_chartqapro
importlib.reload(rose_chartqapro)   # picks up any edits

# Inject already-loaded models (avoid reloading)
rose_chartqapro.run_rose_on_chartqapro_with_models(model, processor, embed_model)

## Cell 7 — View results table

In [ ]:
import json, pandas as pd
from collections import defaultdict

with open('results/rose_cqa_results.json') as f:
    results = json.load(f)

df = pd.DataFrame(results)

# Accuracy by question type and method
summary = df.groupby(['question_type', 'method'])['is_correct'].agg(['mean','count'])
summary['accuracy_%'] = (summary['mean'] * 100).round(1)
print(summary[['accuracy_%', 'count']].to_string())

print(f"\nOverall accuracy: {df['is_correct'].mean()*100:.1f}%")
print(f"Zero-shot accuracy: {df[df.method=='zero_shot_cot']['is_correct'].mean()*100:.1f}%")
print(f"RoSE accuracy: {df[df.method=='rose_few_shot']['is_correct'].mean()*100:.1f}%")

## Cell 8 — Commit results to GitHub

In [ ]:
# Push results back to your GitHub repo automatically
import subprocess

# Set your GitHub credentials
GITHUB_TOKEN = 'YOUR_GITHUB_TOKEN_HERE'   # get from GitHub Settings → Developer Settings → Tokens
GITHUB_USER  = 'ktahsinr'
REPO_NAME    = 'RoSE'

# Copy results into repo
import shutil
shutil.copy('results/rose_cqa_results.json', 'RoSE/results_chartqapro.json')

os.chdir('RoSE')

# Configure git
!git config user.email 'you@northsouth.edu'
!git config user.name  'Yousra'

# Set remote with token
remote = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
!git remote set-url origin {remote}

# Stage, commit, push
!git add results_chartqapro.json rose_chartqapro.py
!git commit -m 'Add RoSE-CQA results on ChartQAPro (all 1948 questions)'
!git push origin main

os.chdir(WORK_DIR)
print('✓ Results pushed to GitHub')